# 01 · Valuations Intro & Pricer Registry

This notebook walks through the **end-to-end pricing lifecycle** in Python using the valuations bindings:

1. Create a pricer registry.
2. Build a simple market data context.
3. Define a fixed-rate bond instrument.
4. Price the bond and inspect NPV plus basic risk metrics (duration, yield, spreads, DV01).
5. Compute **bucketed DV01** (a key-rate duration ladder).
6. Run a **waterfall P&L attribution** for the bond between two dates.

Along the way, you'll see how the registry dispatch works and how deterministic result envelopes are returned.

### Learning Objectives

- Map `finstack.valuations` Python modules to the Rust crate layout.
- Inspect the pricer registry (`InstrumentType`, `ModelKey`) and how dispatch works.
- Create a simple fixed-rate bond, price it, and interpret the returned metadata.
- Compute core risk measures (duration, convexity, yield, spreads, DV01) via `price_with_metrics`.
- Build a **bucketed DV01 ladder** for the bond.
- Run a simple **P&L attribution** between two dates using a waterfall method.

In [1]:
# Standard imports
from datetime import date, timedelta

from finstack import Money
from finstack.core.currency import USD
from finstack.core.market_data.context import MarketContext
from finstack.core.market_data.term_structures import DiscountCurve
from finstack.valuations.instruments import Bond
from finstack.valuations.pricer import create_standard_registry
from finstack.valuations import results
from finstack.valuations.attribution import attribute_pnl, AttributionMethod

import polars as pl

print("Valuations module loaded successfully")

Valuations module loaded successfully


## 1. Architecture Overview

The valuations crate is organized into key modules:
- **instruments**: Bond, IRS, CDS, Options, etc.
- **pricer**: Registry-based pricing dispatch.
- **metrics**: DV01, CS01, Greeks, spreads, and other risk calculators.
- **calibration**: Curve/surface construction.
- **results**: Result envelopes with metadata.
- **risk**: Bucketed DV01 / CS01 ladders.
- **attribution**: P&L attribution between two market snapshots.

In [2]:
# Explore module structure
import finstack.valuations.instruments as inst
import finstack.valuations.pricer as prc
import finstack.valuations.metrics as met
import finstack.valuations.risk as risk
import finstack.valuations.attribution as attr

print("Available instrument types (sample):")
print("  - Bond")
print("  - IRS (Interest Rate Swap)")
print("  - EquityOption")
print("  - CDS (Credit Default Swap)")

print("\nKey helper modules:")
print("  - metrics: standard risk metrics (DV01, duration, spreads, Greeks)")
print("  - risk: bucketed DV01 / CS01 ladders")
print("  - attribution: P&L attribution helpers")

print("\nPricer registry provides deterministic, enum-based dispatch")

Available instrument types (sample):
  - Bond
  - IRS (Interest Rate Swap)
  - EquityOption
  - CDS (Credit Default Swap)

Key helper modules:
  - metrics: standard risk metrics (DV01, duration, spreads, Greeks)
  - risk: bucketed DV01 / CS01 ladders
  - attribution: P&L attribution helpers

Pricer registry provides deterministic, enum-based dispatch


## 2. Registry Walkthrough

The pricer registry maps (InstrumentType, ModelKey) pairs to concrete pricers. This ensures type-safe, deterministic pricing.

In [3]:
# Create standard registry
registry = create_standard_registry()
print("Registry created with all standard pricers registered")
print("\nRegistry contains pricers for:")
print("  - Fixed/floating bonds (discounting)")
print("  - Swaps (par rate, discounting)")
print("  - Options (Black-Scholes, trees, MC)")
print("  - Credit derivatives (ISDA standard)")

Registry created with all standard pricers registered

Registry contains pricers for:
  - Fixed/floating bonds (discounting)
  - Swaps (par rate, discounting)
  - Options (Black-Scholes, trees, MC)
  - Credit derivatives (ISDA standard)


In [4]:
# Build simple market data
# Use an as_of date after issue so accrual and yield metrics are non-zero
as_of = date(2025, 7, 1)
market = MarketContext()

# USD discount curve (flat ~4% for simplicity) with knots aligned to DV01 buckets
usd_disc = DiscountCurve(
    "USD-OIS",
    as_of,
    [
        (0.0, 1.0),      # today
        (0.25, 0.9900),  # 3m
        (0.5, 0.9802),   # 6m
        (1.0, 0.9608),   # 1y
        (2.0, 0.9231),   # 2y
        (3.0, 0.8869),   # 3y
        (5.0, 0.8187),   # 5y
        (7.0, 0.7582),   # 7y
        (10.0, 0.6703),  # 10y
        (15.0, 0.5488),  # 15y
        (20.0, 0.4493),  # 20y
        (30.0, 0.3012),  # 30y
    ],
)
market.insert_discount(usd_disc)

print(f"Market context created with discount curve: USD-OIS")
print(f"DF @ 5Y: {usd_disc.df(5.0):.4f}")

Market context created with discount curve: USD-OIS
DF @ 5Y: 0.8187


In [5]:
# Create a simple fixed-rate bond with a quoted clean price for YTM/spread metrics
bond = Bond.builder(
    "BOND-001",
    Money(1_000_000, USD),
    date(2025, 1, 10),
    date(2030, 1, 15),
    "USD-OIS",
    0.05,     # coupon_rate
    None,     # frequency (defaults to semi-annual)
    None,     # day_count (defaults to 30/360)
    None,     # bdc
    None,     # calendar_id
    None,     # stub
    None,     # amortization
    None,     # call_schedule
    None,     # put_schedule
    99.50,    # quoted_clean_price (% of par)
    None,     # forward_curve
    None,     # float_margin_bp
    None,     # float_gearing
    None,     # float_reset_lag_days
)

print("Created 5-year fixed bond with price override:")
print(f"  ID: {bond.instrument_id}")
print(f"  Notional: {bond.notional}")
print(f"  Coupon: {bond.coupon:.2%}")
print("  Quoted clean price: 99.50")

Created 5-year fixed bond with price override:
  ID: BOND-001
  Notional: USD 1000000.00
  Coupon: 5.00%
  Quoted clean price: 99.50


## 3. Result Envelopes & Metadata

Pricing returns a **result envelope** containing:
- Present value (`value`)
- Metadata (`meta`: numeric mode, rounding context, FX policy)
- Optional risk metrics (`measures`: duration, DV01, spreads, etc.)

We'll start with a plain NPV call and then ask for additional metrics.

In [6]:
# Price the bond at the market's as_of date
result = registry.price(bond, "discounting", market, as_of=as_of)

print(f"Bond NPV: {result.value}")
print("\nResult metadata:")
print(f"  Type: {type(result).__name__}")
print(f"  Currency: {result.value.currency}")
print(f"  Numeric mode: {result.meta.numeric_mode}")
print(f"  FX policy: {result.meta.fx_policy_applied}")

# Results contain deterministic execution context
print("\nDeterministic context preserved in result envelope")

Bond NPV: USD 1063685.22

Result metadata:
  Type: ValuationResult
  Currency: USD
  Numeric mode: f64
  FX policy: None

Deterministic context preserved in result envelope


In [7]:
# Price with metrics (duration, yield, spreads, DV01)
result_with_metrics = registry.price_with_metrics(
    bond,
    "discounting",
    market,
    [
        "accrued",        # accrued interest (Money units)
        "duration_mod",   # modified duration
        "duration_mac",   # Macaulay duration
        "convexity",      # convexity
        "ytm",            # yield to maturity (from quoted price)
        "z_spread",       # zero-vol spread (from quoted price)
        "asw_par",        # par asset swap spread
        "asw_market",     # market asset swap spread (uses quoted price)
        "dv01",           # dollar value of 1bp
    ],
    as_of=as_of,
)

print("Bond valuation with metrics:")
print(f"  NPV: {result_with_metrics.value}")
print("\nMetrics computed:")
for name, value in result_with_metrics.measures.items():
    print(f"  {name}: {value}")

Bond valuation with metrics:
  NPV: USD 1063685.22

Metrics computed:
  accrued: 23750.0
  duration_mod: 3.919151820032632
  duration_mac: 4.01957338958095
  convexity: 18.611707776053574
  ytm: 0.05124658301575138
  z_spread: 0.010662155521653843
  asw_par: 0.009662001923160593
  asw_market: 0.014223981473947607
  dv01: -426.3586712938268


## 5. Simple P&L Attribution Between Two Dates

To understand how the bond's value changes between two days, we can run a **waterfall P&L attribution**:

- Build two market contexts (`market_t0`, `market_t1`) with slightly different discount curves.
- Choose a start and end date (`as_of_t0`, `as_of_t1`).
- Call `attribute_pnl` with an `AttributionMethod.waterfall([...])` factor order.

For this intro, we'll focus on **carry** and **rates_curves** contributions.


In [8]:
# Build two simple market snapshots for attribution
as_of_t0 = as_of
as_of_t1 = as_of + timedelta(days=1)

# T0 market: original discount curve
market_t0 = MarketContext()
usd_disc_t0 = DiscountCurve(
    "USD-OIS",
    as_of_t0,
    [(0.0, 1.0), (1.0, 0.96), (5.0, 0.82), (10.0, 0.67)],
)
market_t0.insert_discount(usd_disc_t0)

# T1 market: slightly lower discount factors (rates up a bit)
market_t1 = MarketContext()
usd_disc_t1 = DiscountCurve(
    "USD-OIS",
    as_of_t1,
    [(0.0, 1.0), (1.0, 0.955), (5.0, 0.81), (10.0, 0.66)],
)
market_t1.insert_discount(usd_disc_t1)

# Run waterfall attribution focusing on carry and rates curve moves
method = AttributionMethod.waterfall(["carry", "rates_curves"])
attr = attribute_pnl(
    bond,
    market_t0,
    market_t1,
    as_of_t0,
    as_of_t1,
    method=method,
)

print(f"Total P&L: {attr.total_pnl}")
print(f"  Carry: {attr.carry}")
print(f"  Rates curves: {attr.rates_curves_pnl}")
print(f"  Residual: {attr.residual} ({attr.meta.residual_pct:.4f}%)")

print("\nAttribution tree (waterfall):")
print(attr.explain())


Total P&L: USD -10815.50
  Carry: USD 122.28
  Rates curves: USD -10937.79
  Residual: USD 0.00 (-0.0000%)

Attribution tree (waterfall):
Total P&L: USD -10815.50
  ├─ Carry: USD 122.28 (-1.1%)
  ├─ Rates Curves: USD -10937.79 (101.1%)
  └─ Residual: USD 0.00 (-0.0%)


## 4. Bucketed DV01 (Key-Rate Duration Ladder)

The `bucketed_dv01` metric computes a **bucketed DV01** profile for the bond:

- It bumps one tenor bucket at a time on the discount curve.
- Reprices the bond under each bump.
- Stores per-bucket values under composite keys like `bucketed_dv01::USD-OIS::5y`.

We fetch those entries from `result.measures` and turn them into a `bucket` / `dv01` table to see which parts of the curve drive rate risk.


In [9]:
# Bucketed DV01 ladder for the bond via the standard `bucketed_dv01` metric
krd_result = registry.price_with_metrics(
    bond,
    "discounting",
    market,
    ["bucketed_dv01"],
    as_of=as_of,
)

krd_result.to_dict()

{'instrument_id': 'BOND-001',
 'as_of': datetime.date(2025, 7, 1),
 'value': Money(amount=1063685.221214, currency='USD'),
 'measures': {'bucketed_dv01': -426.43003050098196,
  'bucketed_dv01::6m': -25.967430935939774,
  'bucketed_dv01::1y': -50.770254293922335,
  'bucketed_dv01::15y': 0.0,
  'bucketed_dv01::3y': -93.31988104968332,
  'bucketed_dv01::7y': 0.0,
  'bucketed_dv01::30y': 0.0,
  'bucketed_dv01::20y': 0.0,
  'bucketed_dv01::2y': -97.93515028222464,
  'bucketed_dv01::3m': -26.02930516260676,
  'bucketed_dv01::5y': -132.40800877660513,
  'bucketed_dv01::10y': 0.0},
 'meta': {'numeric_mode': 'f64',
  'rounding': {'mode': 'bankers',
   'version': 1,
   'ingest_scale_by_ccy': {},
   'output_scale_by_ccy': {}},
  'fx_policy_applied': None,
  'timestamp': '2025-11-16T01:05:00.622299000Z',
  'version': '0.3.0'},
 'covenants': None}

## Summary

Key Takeaways:
1. **Registry-based pricing**: Type-safe dispatch via (InstrumentType, ModelKey).
2. **Result envelopes**: `ValuationResult` wraps PV, metadata, and risk measures.
3. **Core risk metrics**: Duration, convexity, yield, spreads, and DV01 via `price_with_metrics`.
4. **Bucketed DV01**: `bucketed_dv01` exposes a key-rate duration ladder per curve and bucket.
5. **P&L attribution**: `attribute_pnl` decomposes P&L between two dates (e.g., carry vs. rates curves).

Next notebook: Building richer market data contexts and curves.